This document aims to accomplish the following, for every device in the yearly sample (devices seen on 350+ days of the year, [2ai]), using all of its Veraset visits that year (not just dispensary visits), and for each measure the distribution across the sample:

(a) Visit counts: total, home, and non-home visits over the year, and non-home (also total and home) visits per observed day. Label as ([8a])

(b) Gaps between consecutive visits, split into gaps within a day (the next visit starts the same local day) and gaps across days. Within a day, both the gap between start times and the gap from the implied end (start + minimum dwell) to the next start

(i) one row per device, with its gap distribution. Label as ([8bi])

(ii) the pooled distribution of every gap in the sample, as quantiles ([8bii]) and as binned counts for the charts ([8bii'])

(c) Activity over time within each device: is it steady through the year, or concentrated in bursts with long pauses in between? Activity is measured two ways, day by day and week by week: the number of visits, and the expected reciprocal of the gap to the next visit (mean of 1 / gap, in visits per hour). Label as ([8c])

(d) The sample-level distribution (mean, sd, percentiles) of every device-level measure in [8a], [8bi], and [8c], by year and all years. Label as ([8d])

Notes:
- Visits are cleaned the same way as the dispensary visits in Step 4 ([4bi]): duplicates dropped (each UTC month folder repeats the 1st of the next month), visits Veraset cut at UTC midnight (4 or 5 pm local) put back together, and only visits starting inside the device's local calendar year kept. Without this, every midnight cut would show up as a fake within-day gap of a few minutes
- A home visit is one Veraset labels "home" (location_name). "Observed days" are the days in the year with any Veraset record, from Step 4 (348-366 for this sample). Days are local calendar days, weeks run Monday to Sunday
- Gaps and activity are measured twice: over all visits, and over non-home visits only (home visits are mostly one long overnight stay a day, so they make "all visits" look steadier than the rest of the device's activity). This is the "visit_set" column
- Charts also get saved to "Task 2/Charts and Figures" as .html files


In [1]:
### IMPORTS AND PRELIMINARIES
## MEMORY STUFF
# Stops polars from holding onto freed memory, and forces it to be returned to the OS
import os
os.environ['_RJEM_MALLOC_CONF'] = "background_thread:true,dirty_decay_ms:0,muzzy_decay_ms:0"

## WORKHORSE LIBRARY
# Enables data streaming of large files
import polars as pl

## CHARTS (same as week 6)
import numpy as np
import plotly.express as px

# Cleans up some cell outputs
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

YEARS = ["2019", "2020", "2021"]
CHART_FOLDER = "Task 2/Charts and Figures"
os.makedirs(CHART_FOLDER, exist_ok = True)

def show(fig, name):
    fig.show()
    fig.write_html(f"{CHART_FOLDER}/{name}.html", include_plotlyjs = "cdn") # so they can be opened outside the notebook too

windows = pl.read_parquet("processed-data/device_windows_yearly.parquet") # [2ai] devices, their local calendar year window, and observed days (Step 4)
windows.group_by("year").agg(pl.len().alias("devices"), pl.col("observed_days").mean().round(1).alias("mean_observed_days")).sort("year")

<jemalloc>: option background_thread currently supports pthread only


year,devices,mean_observed_days
i32,u32,f64
2019,23552,356.7
2020,38224,358.6
2021,26186,357.9


In [2]:
### HELPER: CLEANING
## Same three fixes as fix_visits in Step 4 ([4bi]), for every visit of one year. The place is the Veraset place ID (home visits have one too)
MIDNIGHT_WINDOW = 1800 # seconds. Veraset picks a cut visit back up with the device's first ping after UTC midnight, which can take ~20-30 minutes
MAX_SPLIT_GAP = 3600 # seconds. How far apart the two halves of a visit cut at UTC midnight can be and still get put back together

def load_visits(YEAR):
    return (pl.scan_parquet(f"processed-data/sample_restricted_yearly_{YEAR}.parquet")
        .select([
            "caid_hash", "utc_timestamp", "local_timestamp", "minimum_dwell",
            pl.col("safegraph_place_id").hash().alias("place"), # a number instead of a long string, to save memory
            (pl.col("location_name") == "home").alias("is_home"),
        ])
        .collect(engine = "streaming")
    )

def clean_visits(visits, year_windows):
    num_rows = visits.shape[0]

    # (1) The same visit twice (the 1st of each month shows up in two month folders): keep one, the longest
    visits = (visits
        .sort("minimum_dwell", descending = True)
        .unique(["caid_hash", "utc_timestamp", "place"], keep = "first")
        .sort(["caid_hash", "utc_timestamp"])
    )
    num_unique = visits.shape[0]

    # (2) A visit starting in the first MIDNIGHT_WINDOW of a UTC day at the same place as the device's previous visit, which started the UTC day before
    #     and ended at most MAX_SPLIT_GAP earlier, is the rest of that visit. The two become one (chains too, for multi-day stays)
    same_device = pl.col("caid_hash") == pl.col("caid_hash").shift(1)
    visits = (visits
        .with_columns((pl.col("utc_timestamp") + (pl.col("minimum_dwell") * 60).cast(pl.Int64)).alias("utc_end"))
        .with_columns(
            (same_device
             & (pl.col("utc_timestamp") % 86400 < MIDNIGHT_WINDOW)
             & (pl.col("place") == pl.col("place").shift(1))
             & (pl.col("utc_timestamp") - pl.col("utc_end").shift(1) <= MAX_SPLIT_GAP)
             & (pl.col("utc_timestamp") // 86400 > pl.col("utc_timestamp").shift(1) // 86400)).fill_null(False).alias("continues")
        )
        .with_columns((~pl.col("continues")).cast(pl.UInt32).cum_sum().alias("visit_number")) # sorted by device, so numbers never cross devices
        .group_by("visit_number", maintain_order = True)
        .agg(pl.col(["caid_hash", "utc_timestamp", "local_timestamp", "place", "is_home"]).first(), pl.col("utc_end").max())
        .with_columns(((pl.col("utc_end") - pl.col("utc_timestamp")) / 60).alias("minimum_dwell"))
        .drop(["visit_number", "utc_end"])
    )
    num_merged = visits.shape[0]

    # (3) Only visits starting inside the device's local calendar year
    visits = (visits
        .with_columns((pl.col("local_timestamp") // 86400).alias("day")) # local day number (local_timestamp is local time written as if it were UTC)
        .join(year_windows.select(["caid_hash", "window_start_day", "window_end_day"]), on = "caid_hash", how = "inner")
        .filter((pl.col("day") >= pl.col("window_start_day")) & (pl.col("day") <= pl.col("window_end_day")))
        .drop(["window_start_day", "window_end_day"])
        .sort(["caid_hash", "utc_timestamp"])
    )
    print(f"rows: {num_rows} | duplicates dropped: {num_rows - num_unique} | halves merged back: {num_unique - num_merged} | outside the year: {num_merged - visits.shape[0]} | visits left: {visits.shape[0]}")
    return visits

In [3]:
### HELPER: GAPS AND ACTIVITY
## Gaps: each visit and the device's next visit (in the same visit set). Within a day = the next one starts the same local day
##   "start_gap" (minutes): next start - this start. "end_gap" (minutes): next start - (this start + minimum dwell); negative when the visits overlap
##   "reciprocal" (visits per hour): 1 / start_gap, with gaps under a minute counted as a minute (Veraset has a few visits starting the same second)
VISIT_SETS = [("all", None), ("non-home", ~pl.col("is_home"))]
QUANTILES = [(0.01, "p1"), (0.05, "p5"), (0.10, "p10"), (0.25, "p25"), (0.5, "median"), (0.75, "p75"), (0.90, "p90"), (0.95, "p95"), (0.99, "p99")]

def add_gaps(visits):
    same_device = pl.col("caid_hash") == pl.col("caid_hash").shift(-1)
    return (visits
        .with_columns(
            pl.when(same_device).then((pl.col("local_timestamp").shift(-1) - pl.col("local_timestamp")) / 60).alias("start_gap"),
            pl.when(same_device).then((pl.col("local_timestamp").shift(-1) - pl.col("local_timestamp")) / 60 - pl.col("minimum_dwell")).alias("end_gap"),
            pl.when(same_device).then(pl.col("day").shift(-1) == pl.col("day")).alias("within_day"),
        )
        .with_columns((60 / pl.col("start_gap").clip(lower_bound = 1)).alias("reciprocal"))
    )

def burstiness(gaps): # (sd - mean) / (sd + mean) of the gaps: -1 for clockwork, about 0 for random (Poisson) timing, toward 1 for bursts and long pauses
    return ((gaps.std() - gaps.mean()) / (gaps.std() + gaps.mean()))

## Day-by-day (and week-by-week) activity, with the days (weeks) of the year without any counted as 0. sd is over every day of the device's year
def spread(values, periods): # mean, sd, and coefficient of variation of a per-period measure, the missing periods being 0
    mean = values.sum() / periods
    sd = ((values.pow(2).sum() / periods) - mean.pow(2)).clip(lower_bound = 0).sqrt()
    return mean, sd, sd / mean

TOP_DAYS = 37 # the busiest 10% of a 365- or 366-day year, rounded up

In [4]:
### PART A, B, C: ONE YEAR AT A TIME
## Output: [8a] one row per sample device and year with its visit counts
##         Columns are "year", "caid_hash", "observed_days", "visits", "home_visits", "non_home_visits", "home_share", "visits_per_observed_day",
##         "home_visits_per_observed_day", "non_home_visits_per_observed_day"
## Name: 8a_device_visit_counts.parquet
## Output: [8bi] one row per sample device, year, and visit set ("all", "non-home") with the distribution of its gaps
##         Columns are "year", "caid_hash", "visit_set", "within_day_gaps", "across_day_gaps", "share_within_day", then for the within-day start-to-start gaps
##         (minutes) "within_start_mean", "within_start_p10", "within_start_median", "within_start_p90", for the within-day end-to-start gaps (minutes)
##         "within_end_mean", "within_end_p10", "within_end_median", "within_end_p90", "within_end_share_overlap" (next visit starts before this one ends),
##         for the across-day gaps (hours, start to start) "across_mean", "across_median", "across_p90", and over every gap "max_gap_hours", "burstiness"
## Name: 8bi_device_gaps.parquet
## Output: [8c] one row per sample device, year, and visit set, with how steady its activity is through the year
##         Columns are "year", "caid_hash", "visit_set", "window_days", "active_days" (days with a visit), "longest_pause_days" (longest run of days without
##         a visit, including the start and end of the year), then for each activity measure ("visits", and "reciprocal" = mean of 1 / gap to the next
##         visit, per hour, over the gaps starting that day): "{measure}_daily_mean", "{measure}_daily_sd", "{measure}_daily_cv" (sd / mean, across every
##         day of the year), "{measure}_weekly_cv" (the same with Monday-Sunday weeks), "{measure}_busiest_10pct_share" (share of the year's activity in its
##         busiest 10% of days; 0.10 if perfectly even)
## Name: 8c_device_activity.parquet
counts_rows, gap_rows, activity_rows = [], [], []
pooled_gaps = [] # every gap, for [8bii]

for YEAR in YEARS:
    year_windows = windows.filter(pl.col("year") == int(YEAR))
    visits = clean_visits(load_visits(YEAR), year_windows)

    # (1) Counts
    counts_rows.append(year_windows.select(["year", "caid_hash", "observed_days"])
        .join(visits.group_by("caid_hash").agg(pl.len().alias("visits"), pl.col("is_home").sum().alias("home_visits")), on = "caid_hash", how = "left")
        .with_columns(pl.col(["visits", "home_visits"]).fill_null(0).cast(pl.Int64))
        .with_columns((pl.col("visits") - pl.col("home_visits")).alias("non_home_visits"))
        .with_columns(
            (pl.col("home_visits") / pl.col("visits")).alias("home_share"),
            *[(pl.col(c) / pl.col("observed_days")).alias(f"{c}_per_observed_day") for c in ["visits", "home_visits", "non_home_visits"]],
        ))

    for visit_set, condition in VISIT_SETS:
        set_visits = add_gaps(visits if condition is None else visits.filter(condition))
        gaps = set_visits.drop_nulls("start_gap")

        # (2) Gaps, device by device
        within = pl.col("within_day")
        gap_rows.append(gaps
            .group_by("caid_hash")
            .agg(
                within.sum().alias("within_day_gaps"),
                (~within).sum().alias("across_day_gaps"),
                within.mean().alias("share_within_day"),
                pl.col("start_gap").filter(within).mean().alias("within_start_mean"),
                *[pl.col("start_gap").filter(within).quantile(q).alias(f"within_start_{n}") for q, n in [(0.10, "p10"), (0.5, "median"), (0.90, "p90")]],
                pl.col("end_gap").filter(within).mean().alias("within_end_mean"),
                *[pl.col("end_gap").filter(within).quantile(q).alias(f"within_end_{n}") for q, n in [(0.10, "p10"), (0.5, "median"), (0.90, "p90")]],
                (pl.col("end_gap").filter(within) < 0).mean().alias("within_end_share_overlap"),
                (pl.col("start_gap").filter(~within) / 60).mean().alias("across_mean"),
                (pl.col("start_gap").filter(~within) / 60).median().alias("across_median"),
                (pl.col("start_gap").filter(~within) / 60).quantile(0.90).alias("across_p90"),
                (pl.col("start_gap").max() / 60).alias("max_gap_hours"),
                burstiness(pl.col("start_gap")).alias("burstiness"),
            )
            .with_columns(pl.lit(int(YEAR)).cast(pl.Int32).alias("year"), pl.lit(visit_set).alias("visit_set")))
        pooled_gaps.append(gaps.select(
            pl.lit(int(YEAR)).cast(pl.Int32).alias("year"), pl.lit(visit_set).alias("visit_set"), "within_day",
            pl.col("start_gap").cast(pl.Float32), pl.col("end_gap").cast(pl.Float32)))

        # (3) Activity, day by day and week by week (a gap belongs to the day/week it starts in)
        set_visits = set_visits.with_columns(((pl.col("day") + 3) // 7).alias("week")) # day 0 (1 Jan 1970) was a Thursday, so weeks start on Monday
        days = year_windows.select(["caid_hash", "window_start_day", "window_end_day",
                                    (pl.col("window_end_day") - pl.col("window_start_day") + 1).alias("window_days"),
                                    ((pl.col("window_end_day") + 3) // 7 - (pl.col("window_start_day") + 3) // 7 + 1).alias("window_weeks")])
        activity = lambda key: (set_visits
            .group_by(["caid_hash", key])
            .agg(pl.len().alias("visits"), pl.col("reciprocal").mean().fill_null(0).alias("reciprocal")) # a device's last visit has no gap
            .join(days, on = "caid_hash"))
        daily, weekly = activity("day"), activity("week")

        daily_stats = (daily
            .group_by("caid_hash")
            .agg(
                pl.len().alias("active_days"),
                (pl.col("day").sort().diff() - 1).max().fill_null(0).alias("pause_inside"),
                (pl.col("day").min() - pl.col("window_start_day").first()).alias("pause_start"),
                (pl.col("window_end_day").first() - pl.col("day").max()).alias("pause_end"),
                *[expr.alias(f"{m}_{name}") for m in ["visits", "reciprocal"]
                  for name, expr in zip(["daily_mean", "daily_sd", "daily_cv"], spread(pl.col(m), pl.col("window_days").first()))],
                *[(pl.col(m).top_k(TOP_DAYS).sum() / pl.col(m).sum()).alias(f"{m}_busiest_10pct_share") for m in ["visits", "reciprocal"]],
            )
            .with_columns(pl.max_horizontal("pause_inside", "pause_start", "pause_end").alias("longest_pause_days"))
        )
        weekly_stats = (weekly
            .group_by("caid_hash")
            .agg([spread(pl.col(m), pl.col("window_weeks").first())[2].alias(f"{m}_weekly_cv") for m in ["visits", "reciprocal"]]))
        activity_rows.append(days.select(["caid_hash", "window_days"])
            .join(daily_stats, on = "caid_hash", how = "left")
            .join(weekly_stats, on = "caid_hash", how = "left")
            .with_columns(pl.col("active_days").fill_null(0), pl.lit(int(YEAR)).cast(pl.Int32).alias("year"), pl.lit(visit_set).alias("visit_set")))

    del visits, set_visits, gaps, daily, weekly

device_counts = pl.concat(counts_rows).sort(["year", "caid_hash"])
device_counts.write_parquet("processed-data/8a_device_visit_counts.parquet")

GAP_COLUMNS = ["within_day_gaps", "across_day_gaps", "share_within_day", "within_start_mean", "within_start_p10", "within_start_median", "within_start_p90",
               "within_end_mean", "within_end_p10", "within_end_median", "within_end_p90", "within_end_share_overlap", "across_mean", "across_median",
               "across_p90", "max_gap_hours", "burstiness"]
device_gaps = pl.concat(gap_rows).select(["year", "caid_hash", "visit_set"] + GAP_COLUMNS).sort(["year", "visit_set", "caid_hash"])
device_gaps.write_parquet("processed-data/8bi_device_gaps.parquet")

ACTIVITY_COLUMNS = ["window_days", "active_days", "longest_pause_days"] + [f"{m}_{s}" for m in ["visits", "reciprocal"]
                                                                          for s in ["daily_mean", "daily_sd", "daily_cv", "weekly_cv", "busiest_10pct_share"]]
device_activity = pl.concat(activity_rows).select(["year", "caid_hash", "visit_set"] + ACTIVITY_COLUMNS).sort(["year", "visit_set", "caid_hash"])
device_activity.write_parquet("processed-data/8c_device_activity.parquet")

pooled_gaps = pl.concat(pooled_gaps)
print("devices:", device_counts.shape[0], "| gap rows:", device_gaps.shape[0], "| activity rows:", device_activity.shape[0], "| pooled gaps:", pooled_gaps.shape[0])
device_counts.head()

rows: 30860914 | duplicates dropped: 922712 | halves merged back: 4347092 | outside the year: 67176 | visits left: 25523934


rows: 44789098 | duplicates dropped: 2853766 | halves merged back: 8250168 | outside the year: 95667 | visits left: 33589497


rows: 34168208 | duplicates dropped: 1009473 | halves merged back: 5392610 | outside the year: 72551 | visits left: 27693574


devices: 87962 | gap rows: 175754 | activity rows: 175924 | pooled gaps: 135867575


year,caid_hash,observed_days,visits,home_visits,non_home_visits,home_share,visits_per_observed_day,home_visits_per_observed_day,non_home_visits_per_observed_day
i32,u64,i64,i64,i64,i64,f64,f64,f64,f64
2019,2007570705863234,354,1183,543,640,0.459003,3.341808,1.533898,1.80791
2019,3416136947360443,363,771,404,367,0.523995,2.123967,1.112948,1.011019
2019,4321084926670314,365,1631,758,873,0.464746,4.468493,2.076712,2.391781
2019,4436985526214702,365,1371,535,836,0.390226,3.756164,1.465753,2.290411
2019,4556691208931819,360,530,350,180,0.660377,1.472222,0.972222,0.5


In [5]:
### PART B, SUBPART II
## Output: [8bii] the pooled distribution of every gap in the sample, by visit set, gap type, and year (and "all")
##         Gap types are "within day, start to start" (minutes), "within day, end to start" (minutes, negative = overlap), "across days, start to start" (hours)
##         Columns are "visit_set", "gap_type", "period", "gaps", "share_negative", "mean", "sd", "min", "p1" ... "p99", "max"
## Name: 8bii_pooled_gaps.parquet
## Output: [8bii'] the same gaps in bins, for the charts. Columns are "visit_set", "gap_type", "bin_order", "bin", "gaps", "share" (of that set and type)
## Name: 8bii_pooled_gap_bins.parquet
GAP_TYPES = [
    ("within day, start to start", pl.col("within_day"), pl.col("start_gap")),
    ("within day, end to start", pl.col("within_day"), pl.col("end_gap")),
    ("across days, start to start", ~pl.col("within_day"), pl.col("start_gap") / 60),
]
BINS = { # (edges, labels): minutes for within-day gaps, hours across days
    "within day, start to start": ([0, 5, 15, 30, 60, 120, 240, 480, 1440], ["<5 min", "5-15 min", "15-30 min", "30-60 min", "1-2 h", "2-4 h", "4-8 h", "8-24 h"]),
    "within day, end to start": ([-1e9, -60, 0, 5, 15, 30, 60, 120, 240, 1440],
                                 ["overlap >1 h", "overlap <1 h", "0-5 min", "5-15 min", "15-30 min", "30-60 min", "1-2 h", "2-4 h", "4-24 h"]),
    "across days, start to start": ([0, 6, 12, 18, 24, 36, 48, 72, 168, 1e9], ["<6 h", "6-12 h", "12-18 h", "18-24 h", "24-36 h", "36-48 h", "2-3 days", "3-7 days", "7+ days"]),
}

quantile_rows, bin_rows = [], []
for visit_set, _ in VISIT_SETS:
    for gap_type, condition, value in GAP_TYPES:
        rows = pooled_gaps.filter((pl.col("visit_set") == visit_set) & condition).select("year", value.alias("gap"))
        for label, part in [(YEAR, rows.filter(pl.col("year") == int(YEAR))) for YEAR in YEARS] + [("all", rows)]:
            quantile_rows.append(part.select(
                pl.lit(visit_set).alias("visit_set"), pl.lit(gap_type).alias("gap_type"), pl.lit(label).alias("period"),
                pl.len().alias("gaps"), (pl.col("gap") < 0).mean().alias("share_negative"),
                pl.col("gap").mean().alias("mean"), pl.col("gap").std().alias("sd"), pl.col("gap").min().alias("min"),
                *[pl.col("gap").quantile(q).alias(n) for q, n in QUANTILES], pl.col("gap").max().alias("max")))
        edges, labels = BINS[gap_type]
        binned = (rows
            .select(pl.col("gap").cut(edges[1:-1], labels = labels, left_closed = True).alias("bin"))
            .group_by("bin").len().rename({"len": "gaps"}))
        bin_rows.append(pl.DataFrame({"bin": labels, "bin_order": range(len(labels))}, schema = {"bin": pl.String, "bin_order": pl.Int32})
            .join(binned.with_columns(pl.col("bin").cast(pl.String)), on = "bin", how = "left")
            .with_columns(pl.col("gaps").fill_null(0))
            .with_columns((pl.col("gaps") / pl.col("gaps").sum()).alias("share"), pl.lit(visit_set).alias("visit_set"), pl.lit(gap_type).alias("gap_type"))
            .select(["visit_set", "gap_type", "bin_order", "bin", "gaps", "share"]))

pooled_gap_quantiles = pl.concat(quantile_rows, how = "vertical_relaxed")
pooled_gap_quantiles.write_parquet("processed-data/8bii_pooled_gaps.parquet")
pooled_gap_bins = pl.concat(bin_rows)
pooled_gap_bins.write_parquet("processed-data/8bii_pooled_gap_bins.parquet")

pooled_gap_quantiles.filter(pl.col("period") == "all").with_columns(pl.exclude(["visit_set", "gap_type", "period", "gaps"]).round(2))

visit_set,gap_type,period,gaps,share_negative,mean,sd,min,p1,p5,p10,p25,median,p75,p90,p95,p99,max
str,str,str,u32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""all""","""within day, start to start""","""all""",61307466,0.0,136.649994,162.240005,-54.700001,6.55,11.63,16.549999,31.75,72.32,176.770004,350.970001,503.579987,735.280029,1439.150024
"""all""","""within day, end to start""","""all""",61307466,0.01,56.889999,97.220001,-1439.97998,0.43,2.85,5.08,11.1,23.879999,59.669998,144.529999,244.220001,511.970001,1418.77002
"""all""","""across days, start to start""","""all""",25411578,0.0,24.65,26.67,0.01,2.96,9.52,11.72,14.88,19.540001,24.120001,43.740002,57.439999,119.519997,6215.910156
"""non-home""","""within day, start to start""","""all""",29527412,0.0,140.470001,161.630005,-54.700001,5.47,10.47,15.53,31.4,75.129997,190.830002,364.220001,495.929993,713.830017,1439.920044
"""non-home""","""within day, end to start""","""all""",29527412,0.01,88.050003,135.039993,-1436.72998,0.18,2.1,4.47,11.4,30.42,105.400002,258.829987,371.649994,630.969971,1430.430054
"""non-home""","""across days, start to start""","""all""",19621119,0.0,35.110001,48.84,0.01,6.9,12.26,14.09,17.68,22.940001,39.740002,68.860001,95.839996,190.490005,7268.689941


In [6]:
### PART D
## Output: [8d] the distribution across sample devices of every device-level measure, by year and all years pooled (device-years)
##         Columns are "table" ("8a", "8bi", "8c"), "visit_set" ("all", "non-home", or "-" for [8a]), "measure", "period", "devices" (with a value),
##         "mean", "sd", "min", "p1", "p5", "p10", "p25", "median", "p75", "p90", "p95", "p99", "max"
## Name: 8d_sample_distributions.parquet
COUNT_COLUMNS = ["observed_days", "visits", "home_visits", "non_home_visits", "home_share", "visits_per_observed_day", "home_visits_per_observed_day",
                 "non_home_visits_per_observed_day"]

long_tables = [
    device_counts.with_columns(pl.lit("8a").alias("table"), pl.lit("-").alias("visit_set"))
        .unpivot(index = ["table", "visit_set", "year", "caid_hash"], on = COUNT_COLUMNS, variable_name = "measure"),
    device_gaps.with_columns(pl.lit("8bi").alias("table"))
        .unpivot(index = ["table", "visit_set", "year", "caid_hash"], on = GAP_COLUMNS, variable_name = "measure"),
    device_activity.with_columns(pl.lit("8c").alias("table"))
        .unpivot(index = ["table", "visit_set", "year", "caid_hash"], on = ACTIVITY_COLUMNS, variable_name = "measure"),
]
long_values = pl.concat([t.with_columns(pl.col("value").cast(pl.Float64)) for t in long_tables]).drop_nulls("value").filter(pl.col("value").is_finite())

def describe(group):
    return group.agg(
        pl.len().alias("devices"), pl.col("value").mean().alias("mean"), pl.col("value").std().alias("sd"), pl.col("value").min().alias("min"),
        *[pl.col("value").quantile(q).alias(n) for q, n in QUANTILES], pl.col("value").max().alias("max"))

KEYS = ["table", "visit_set", "measure"]
distributions = pl.concat([
    describe(long_values.with_columns(pl.col("year").cast(pl.String).alias("period")).group_by(KEYS + ["period"], maintain_order = True)),
    describe(long_values.group_by(KEYS, maintain_order = True)).with_columns(pl.lit("all").alias("period")),
], how = "diagonal_relaxed").select(KEYS + ["period", "devices", "mean", "sd", "min"] + [n for _, n in QUANTILES] + ["max"]).sort(KEYS + ["period"], maintain_order = True)
distributions.write_parquet("processed-data/8d_sample_distributions.parquet")

# (1) Visit counts, all years pooled
distributions.filter((pl.col("table") == "8a") & (pl.col("period") == "all")).with_columns(pl.col(pl.Float64).round(2))

table,visit_set,measure,period,devices,mean,sd,min,p1,p5,p10,p25,median,p75,p90,p95,p99,max
str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""8a""","""-""","""home_share""","""all""",87961,0.47,0.12,0.03,0.21,0.29,0.33,0.39,0.46,0.54,0.62,0.69,0.83,1.0
"""8a""","""-""","""home_visits""","""all""",87962,427.12,161.78,0.0,81.0,165.0,223.0,325.0,422.0,522.0,627.0,699.0,868.0,1850.0
"""8a""","""-""","""home_visits_per_observed_day""","""all""",87962,1.19,0.45,0.0,0.23,0.46,0.62,0.91,1.18,1.46,1.75,1.95,2.43,5.08
"""8a""","""-""","""non_home_visits""","""all""",87962,559.75,368.4,0.0,32.0,116.0,178.0,309.0,494.0,726.0,1002.0,1211.0,1820.0,7526.0
"""8a""","""-""","""non_home_visits_per_observed_d…","""all""",87962,1.56,1.03,0.0,0.09,0.32,0.5,0.86,1.38,2.03,2.79,3.38,5.06,20.56
"""8a""","""-""","""observed_days""","""all""",87962,357.85,5.29,348.0,348.0,349.0,350.0,353.0,358.0,363.0,365.0,365.0,366.0,366.0
"""8a""","""-""","""visits""","""all""",87962,986.87,478.11,0.0,138.0,312.0,434.0,663.0,935.0,1241.0,1573.0,1819.0,2454.0,8525.0
"""8a""","""-""","""visits_per_observed_day""","""all""",87962,2.76,1.33,0.0,0.38,0.87,1.21,1.85,2.62,3.47,4.4,5.08,6.83,23.29


In [7]:
# (2) The same, year by year: total, home, and non-home visits, and non-home visits per observed day (medians and means)
(distributions
    .filter((pl.col("table") == "8a") & pl.col("measure").is_in(["visits", "home_visits", "non_home_visits", "non_home_visits_per_observed_day"]))
    .select(["measure", "period", "mean", "p10", "median", "p90"])
    .with_columns(pl.col(pl.Float64).round(2)))

measure,period,mean,p10,median,p90
str,str,f64,f64,f64,f64
"""home_visits""","""2019""",472.89,300.0,466.0,659.0
"""home_visits""","""2020""",389.48,191.0,389.0,579.0
"""home_visits""","""2021""",440.91,228.0,430.0,655.0
"""home_visits""","""all""",427.12,223.0,422.0,627.0
"""non_home_visits""","""2019""",610.84,235.0,558.0,1032.0
…,…,…,…,…,…
"""non_home_visits_per_observed_d…","""all""",1.56,0.5,1.38,2.79
"""visits""","""2019""",1083.73,578.0,1045.0,1619.0
"""visits""","""2020""",878.75,368.0,828.0,1418.0


In [8]:
# (3) Gaps: each device's typical gap, across the sample (all years pooled)
distributions.filter((pl.col("table") == "8bi") & (pl.col("period") == "all")).with_columns(pl.col(pl.Float64).round(3))

table,visit_set,measure,period,devices,mean,sd,min,p1,p5,p10,p25,median,p75,p90,p95,p99,max
str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""8bi""","""all""","""across_day_gaps""","""all""",87960,288.899,67.04,1.0,76.0,145.0,187.0,259.0,313.0,339.0,349.0,354.0,360.0,365.0
"""8bi""","""all""","""across_mean""","""all""",87960,29.511,46.716,1.448,12.815,15.373,16.805,19.467,23.209,30.094,43.637,57.54,110.396,5976.474
"""8bi""","""all""","""across_median""","""all""",87960,22.56,37.997,0.878,10.727,13.652,14.802,17.277,20.399,23.176,25.582,38.681,69.131,5976.474
"""8bi""","""all""","""across_p90""","""all""",87960,52.608,93.324,2.196,18.616,21.833,23.498,25.653,39.342,51.471,91.897,121.122,254.515,6191.032
"""8bi""","""all""","""burstiness""","""all""",87956,0.183,0.105,-0.626,-0.066,0.017,0.054,0.115,0.182,0.251,0.312,0.351,0.45,0.786
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""8bi""","""non-home""","""within_end_share_overlap""","""all""",87663,0.01,0.017,0.0,0.0,0.0,0.0,0.0,0.002,0.016,0.031,0.041,0.067,1.0
"""8bi""","""non-home""","""within_start_mean""","""all""",87663,142.499,52.548,0.083,51.4,70.887,83.46,106.925,136.475,169.115,206.699,235.134,307.505,938.4
"""8bi""","""non-home""","""within_start_median""","""all""",87663,87.598,48.477,0.083,29.717,38.883,44.517,56.783,76.75,105.133,139.3,168.367,265.067,1251.25


In [9]:
# (4) Activity over time: how steady each device is, across the sample (all years pooled)
distributions.filter((pl.col("table") == "8c") & (pl.col("period") == "all")).with_columns(pl.col(pl.Float64).round(3))

table,visit_set,measure,period,devices,mean,sd,min,p1,p5,p10,p25,median,p75,p90,p95,p99,max
str,str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""8c""","""all""","""active_days""","""all""",87962,289.893,67.054,0.0,77.0,146.0,188.0,260.0,314.0,340.0,350.0,355.0,361.0,366.0
"""8c""","""all""","""longest_pause_days""","""all""",87961,6.976,8.852,0.0,1.0,2.0,2.0,3.0,5.0,8.0,12.0,17.0,40.0,330.0
"""8c""","""all""","""reciprocal_busiest_10pct_share""","""all""",87960,0.459,0.16,0.134,0.215,0.254,0.279,0.335,0.434,0.55,0.67,0.769,0.958,1.0
"""8c""","""all""","""reciprocal_daily_cv""","""all""",87960,1.914,1.217,0.325,0.602,0.758,0.855,1.07,1.567,2.445,3.34,4.034,6.063,19.105
"""8c""","""all""","""reciprocal_daily_mean""","""all""",87961,0.673,0.347,0.0,0.061,0.185,0.268,0.425,0.628,0.872,1.135,1.312,1.681,2.89
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""8c""","""non-home""","""visits_daily_cv""","""all""",87828,1.268,0.854,0.305,0.551,0.654,0.719,0.858,1.083,1.421,1.91,2.375,4.15,19.105
"""8c""","""non-home""","""visits_daily_mean""","""all""",87828,1.534,1.008,0.003,0.104,0.323,0.492,0.849,1.353,1.989,2.742,3.318,4.984,20.563
"""8c""","""non-home""","""visits_daily_sd""","""all""",87828,1.504,0.602,0.052,0.403,0.718,0.876,1.137,1.434,1.774,2.16,2.483,3.564,9.784


In [10]:
### CHARTS
## Histograms are binned here (numpy) and drawn as bars, so the saved files stay small
def histogram(df, column, bins, group = None):
    rows = []
    for name, part in ([(None, df)] if group is None else df.group_by(group, maintain_order = True)):
        values = part[column].drop_nulls().to_numpy()
        values = values[np.isfinite(values)]
        counts, edges = np.histogram(np.clip(values, bins[0], bins[-1]), bins = bins)
        rows.append(pl.DataFrame({"bin_start": edges[:-1], "devices": counts, "share": counts / max(counts.sum(), 1)})
                    .with_columns(pl.lit(None if name is None else name[0]).alias("group")))
    return pl.concat(rows)

# (1) Visits per device over the year: total, home, non-home
plot_df = histogram(device_counts.unpivot(index = ["year", "caid_hash"], on = ["visits", "home_visits", "non_home_visits"], variable_name = "type"),
                    "value", np.arange(0, 3050, 50), group = "type")
fig1 = px.line(
    plot_df.with_columns(pl.col("share") * 100).to_pandas(),
    x="bin_start",
    y="share",
    color="group",
    labels={
        "bin_start": "Visits in the year (bins of 50; the last bin is 3,000+)",
        "share": "Share of device-years (%)",
        "group": "",
    },
    title="Visits per Device-Year: Total, Home, and Non-Home, Yearly Sample 2019-2021",
)
show(fig1, "device_visits_total_home_non_home")

# (2) Non-home visits per observed day
plot_df = histogram(device_counts, "non_home_visits_per_observed_day", np.arange(0, 10.25, 0.25))
fig2 = px.bar(
    plot_df.with_columns(pl.col("share") * 100).to_pandas(),
    x="bin_start",
    y="share",
    labels={
        "bin_start": "Non-home visits per observed day (bins of 0.25; the last bin is 10+)",
        "share": "Share of device-years (%)",
    },
    title="Non-Home Visits per Observed Day, Yearly Sample 2019-2021",
)
show(fig2, "non_home_visits_per_observed_day")

In [11]:
# (3) Every gap in the sample, by type (pooled)
for gap_type, name in [("within day, start to start", "within_day_start"), ("within day, end to start", "within_day_end"), ("across days, start to start", "across_days")]:
    fig3 = px.bar(
        pooled_gap_bins.filter(pl.col("gap_type") == gap_type).sort("bin_order").with_columns(pl.col("share") * 100).to_pandas(),
        x="bin",
        y="share",
        color="visit_set",
        barmode="group",
        labels={
            "bin": "Gap to the next visit",
            "share": "Share of gaps (%)",
            "visit_set": "Visits",
        },
        title=f"Gaps Between Visits: {gap_type.capitalize()}, Yearly Sample 2019-2021",
    )
    show(fig3, f"gaps_{name}")

In [12]:
# (4) Activity over time: day-to-day variation in each device's activity (coefficient of variation across the days of its year), both measures
plot_df = histogram(
    device_activity.unpivot(index = ["year", "caid_hash", "visit_set"], on = ["visits_daily_cv", "reciprocal_daily_cv"], variable_name = "measure")
        .with_columns((pl.col("visit_set") + ", " + pl.col("measure").str.replace("_daily_cv", "").str.replace("reciprocal", "1 / gap")).alias("series")),
    "value", np.arange(0, 4.05, 0.05), group = "series")
fig4 = px.line(
    plot_df.with_columns(pl.col("share") * 100).to_pandas(),
    x="bin_start",
    y="share",
    color="group",
    labels={
        "bin_start": "Day-to-day coefficient of variation of activity (sd / mean; the last bin is 4+)",
        "share": "Share of device-years (%)",
        "group": "Visits, activity measure",
    },
    title="How Steady Is Each Device's Activity? Day-to-Day Variation, Yearly Sample 2019-2021",
)
show(fig4, "activity_daily_cv")

# (5) Burstiness of the gaps
plot_df = histogram(device_gaps, "burstiness", np.arange(-1, 1.025, 0.025), group = "visit_set")
fig5 = px.line(
    plot_df.with_columns(pl.col("share") * 100).to_pandas(),
    x="bin_start",
    y="share",
    color="group",
    labels={
        "bin_start": "Burstiness of gaps between visits ((sd - mean) / (sd + mean): -1 regular, 0 random, 1 bursty)",
        "share": "Share of device-years (%)",
        "group": "Visits",
    },
    title="Burstiness of Each Device's Visits, Yearly Sample 2019-2021",
)
show(fig5, "activity_burstiness")

In [13]:
### CHECKS
# (1) Counts add up, and every sample device is there once a year
print("[8a] rows = sample device-years:", device_counts.shape[0] == windows.shape[0],
      "| home + non-home = total:", (device_counts["home_visits"] + device_counts["non_home_visits"] == device_counts["visits"]).all())
# (2) Gap counts: a device with n visits in a set has n - 1 gaps
gap_check = (device_gaps.filter(pl.col("visit_set") == "all")
    .join(device_counts, on = ["year", "caid_hash"])
    .select(((pl.col("within_day_gaps") + pl.col("across_day_gaps")) == pl.col("visits") - 1).all()))
print("gaps = visits - 1 for every device:", gap_check.item())
# (3) Activity: mean visits per day x days in the year = visits
activity_check = (device_activity.filter(pl.col("visit_set") == "all")
    .join(device_counts, on = ["year", "caid_hash"])
    .select(((pl.col("visits_daily_mean") * pl.col("window_days") - pl.col("visits")).abs() < 1e-6).all()))
print("daily mean x days = visits:", activity_check.item())
# (4) Every device is active on at least its observed days with a visit, and on no more days than the year has
print("active days <= window days:", (device_activity["active_days"] <= device_activity["window_days"]).all())

[8a] rows = sample device-years: True | home + non-home = total: True
gaps = visits - 1 for every device: True
daily mean x days = visits: True
active days <= window days: True


### NOTES
- Cleaning matters a lot here: of 109.8M raw rows, 4.8M are duplicates (mostly the 1st of each month, twice; 2020 has the most, 6% of its rows) and 18.0M are halves of visits Veraset cut at UTC midnight (mostly home stays that run through 4-5 pm local). 86.8M visits are left.
- Visit counts (device-years, median [10th-90th percentile]): 935 visits a year [434-1,573], of which 422 home [223-627] and 494 non-home [178-1,002]. Home visits are 46% of a device's visits at the median. Per observed day: 2.6 visits, 1.2 home, 1.4 non-home. 2020 is lower across the board (median 828 visits, 1.18 non-home per observed day, against 1,045 and 1.57 in 2019), which lines up with the pandemic.
- Gaps (all visits, pooled): 71% of gaps are within a day. Within a day, the median time from one start to the next is 72 minutes (10th-90th: 17 min-5.8 h), and from the implied end of one visit to the next start 24 minutes (5 min-2.4 h). 1% of within-day visits overlap the next one (it starts before the implied end). Across days, the median start-to-start gap is 19.5 hours, and 7% are longer than two days (16% of non-home gaps).
- Activity over time: recording is steady at the scale of weeks and mildly bursty at the scale of days. For all visits, the median device-year's daily counts have a coefficient of variation of 0.75, against about 0.63 if the same number of visits came at random (Poisson) times, and 0.36 week to week. The busiest 10% of days hold 26% of its visits. Non-home visits are lumpier: daily CV 1.08 (Poisson about 0.86), weekly 0.50, busiest 10% of days 34%, and a longest stretch with no non-home visit of 8 days at the median (24 at the 90th percentile, 103 at the 99th).
- The reciprocal-gap measure (mean of 1 / gap, visits per hour) is far more concentrated than the count, since a few days with visits minutes apart dominate it: the busiest 10% of days hold 43% of it for all visits and 67% for non-home visits, and its day-to-day CV is 1.6 (all) / 2.6 (non-home) at the median. Burstiness of the gaps is 0.18 (all) / 0.16 (non-home) at the median: above random timing (0), well short of strongly bursty (1).
- Devices start a visit on 314 days of the year at the median (all visits), against 358 observed days: on the other ~44 observed days the device's only records are continuing stays (mostly at home) that started the day before.
- Long pauses can't happen at the day level by construction (the sample needs 350+ observed days), so "pauses" here are days without a visit starting, not days without data.
